<a href="https://colab.research.google.com/github/Gabriel0825/ChatBot.py/blob/main/FraudGuard_HackConRD_Explicado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ FraudGuard AI — Detector de Fraude Bancario
## HackConRD · Construye tu propio detector desde cero

---

> **Cómo usar este cuaderno:**
> Sigue los pasos en orden. Ejecuta cada celda con el botón ▶️ o con `Shift + Enter`.
> Cada paso tiene una explicación completa antes del código.
> **No te saltes ninguno** — cada paso depende del anterior.

---

### 🎬 La historia de este cuaderno

Cada vez que pasas tu tarjeta en una tienda, el banco tiene **menos de 300 milisegundos** para decidir:

> *¿Esta compra la hizo realmente el dueño de la tarjeta,  
> o es un fraude que hay que bloquear ahora mismo?*

Esa decisión no la toma un humano — la toma un programa de computadora  
que en menos tiempo del que demora un parpadeo analiza 30 variables  
y da un veredicto automático.

**Hoy vas a construir ese programa exactamente.**  
Con 50,000 transacciones bancarias reales.  
Con los mismos algoritmos que usan los bancos modernos.  
Sin experiencia previa. Desde cero.

---

### 📋 Lo que vas a hacer en 9 pasos

| Paso | Qué harás | Por qué importa |
|------|-----------|-----------------|
| 1 | Instalar herramientas especiales | Sin esto, nada funciona |
| 2 | Abrir las librerías de ML | Cargar las herramientas en memoria |
| 3 | Cargar 50,000 transacciones reales | Estos son los datos con los que aprende el modelo |
| 4 | Escalar los números | Para que ninguna variable tenga más peso solo por ser más grande |
| 5 | Dividir datos en Entrenamiento y Prueba | **La regla de oro — no te la saltes** |
| 6 | Entrenar los 3 modelos | El momento en que la máquina aprende |
| 7 | Evaluar los resultados | ¿Cuántos fraudes detectó? |
| 8 | Ver transacciones reales juzgadas | El modelo en acción |
| 9 | Crear tu propia transacción | Tú decides los valores — el modelo decide el veredicto |

---


## 💻 ¿Cómo ejecutar este cuaderno?

---

### Opción A — Google Colab ✅ Recomendado para la charla

Ya estás aquí. No necesitas instalar nada.

1. Haz clic en **Archivo → Guardar copia en Drive**
2. Eso crea **tu propia copia** en tu Google Drive — lo que hagas no afecta el original
3. Ejecuta cada celda en orden con **Shift + Enter** o el botón ▶️

> Google Colab te da una máquina virtual con Python ya instalado, GPU gratuita y todas las librerías básicas disponibles. Es la forma más rápida de empezar.

---

### Opción B — En tu propia máquina 🖥️

Si prefieres correrlo en tu computadora, necesitas instalar algunas cosas primero.

#### Requisitos previos

| Requisito | Versión mínima | Dónde descargarlo |
|-----------|---------------|-------------------|
| Python    | 3.9 o superior | [python.org/downloads](https://python.org/downloads) |
| pip       | Viene con Python | — |
| Jupyter   | Cualquiera | Se instala abajo |

> Para verificar que tienes Python instalado, abre la terminal (CMD en Windows, Terminal en Mac) y escribe: `python --version`

#### Paso 1 — Abrir la terminal

- **Windows:** Tecla Windows → escribe `cmd` → Enter
- **Mac:** Command + Espacio → escribe `terminal` → Enter
- **Linux:** Ctrl + Alt + T

#### Paso 2 — Instalar todas las dependencias

Copia este comando completo y pégalo en la terminal:

```bash
pip install pandas numpy scikit-learn imbalanced-learn xgboost jupyter notebook
```

Espera a que termine — puede tardar 2 a 5 minutos dependiendo de tu conexión.

#### Paso 3 — Descargar el dataset

El cuaderno lo descarga automáticamente desde GitHub.  
Si prefieres tenerlo localmente, descárgalo aquí:

```
https://raw.githubusercontent.com/juanmiguelsuero/fraudguard-ai/main/creditcard_mini.csv
```

Guárdalo en la misma carpeta donde está este cuaderno `.ipynb`.

#### Paso 4 — Descargar este cuaderno

Si llegaste aquí por el QR, ya tienes el link.  
Descarga el archivo `.ipynb` y guárdalo en una carpeta de tu computadora.

#### Paso 5 — Abrir el cuaderno

En la terminal, navega a la carpeta donde guardaste el archivo:

```bash
cd C:\Users\TuNombre\Documents\HackConRD
```

Luego escribe:

```bash
jupyter notebook
```

Se abre el navegador automáticamente con Jupyter.  
Haz clic en el archivo `FraudGuard_HackConRD_Explicado.ipynb` para abrirlo.

#### Paso 6 — Ejecutar

Ejecuta cada celda en orden con **Shift + Enter**.  
El resultado aparece justo debajo de cada celda.

---

> 💡 **¿Tienes algún error?** Los más comunes:
> - `ModuleNotFoundError` → te faltó instalar alguna librería del Paso 2
> - `FileNotFoundError` → el dataset no está en la misma carpeta que el cuaderno
> - `python no reconocido` → Python no está instalado o no está en el PATH

---


## 📦 Paso 1 de 9 — Instalar las herramientas que faltan

---

### 🎯 Por qué hacemos esto

Google Colab viene con muchas herramientas pre-instaladas.  
Pero nos faltan **dos específicas** que hay que instalar aparte:

| Herramienta | Para qué la necesitamos |
|-------------|------------------------|
| `imbalanced-learn` | Crear ejemplos extra de fraude (SMOTE) — el dataset tiene muy pocos fraudes naturales |
| `xgboost` | Uno de los tres algoritmos que vamos a entrenar |

**La analogía:** es como ir a cocinar a casa de un amigo.  
La cocina tiene lo básico — cuchillo, sartén, platos.  
Pero tú necesitas dos ingredientes especiales que él no tiene, así que los traes tú.

### 💬 Qué hace el código línea por línea

```
!pip install ...
```
El `!` significa: *ejecuta esto en la terminal, no en Python*.  
`pip` es el instalador de paquetes de Python.  
`--quiet` le dice que instale sin llenar la pantalla de texto innecesario.

> ⏱️ **Tiempo estimado:** 15–30 segundos.  
> Cuando veas el `✅` puedes continuar al Paso 2.


In [ ]:
# El símbolo ! le dice a Colab: ejecuta esto en la terminal, no en Python
# --quiet = instala en silencio, sin llenar la pantalla de mensajes
!pip install imbalanced-learn xgboost --quiet

# Confirmación de que todo salió bien
print('✅ Herramientas instaladas correctamente.')
print('   Continúa al Paso 2 →')


✅ Herramientas instaladas correctamente.
   Continúa al Paso 2 →


## 🔧 Paso 2 de 9 — Abrir las librerías de Machine Learning

---

### 🎯 Por qué hacemos esto

Instalar una herramienta no es lo mismo que tenerla lista para usar.

**La analogía:** tener la caja de herramientas en el maletero del carro.  
Sabes que está ahí. Pero para trabajar, tienes que sacarla  
y ponerla sobre la mesa. Eso es lo que hace `import`.

Cada línea de `import` abre una herramienta y la deja disponible  
para todo el código que viene después.

---

### 💬 Qué abre cada `import`

| Línea | Herramienta | Para qué la usamos |
|-------|-------------|-------------------|
| `import pandas as pd` | Pandas | Leer y manipular la tabla de datos — como Excel pero en código |
| `import numpy as np` | NumPy | Cálculos matemáticos rápidos con arrays de números |
| `warnings.filterwarnings('ignore')` | — | Silenciar avisos no críticos para tener la pantalla limpia |
| `train_test_split` | Scikit-learn | Dividir datos en 80% entrenamiento / 20% prueba |
| `StandardScaler` | Scikit-learn | Poner todos los números en la misma escala |
| `roc_auc_score` | Scikit-learn | Medir qué tan bueno es el modelo (1.0 = perfecto, 0.5 = azar) |
| `confusion_matrix` | Scikit-learn | Ver cuántos fraudes detectó y cuántos se escaparon |
| `RandomForestClassifier` | Scikit-learn | Algoritmo #1: 20 árboles de decisión votando juntos |
| `LogisticRegression` | Scikit-learn | Algoritmo #2: una línea matemática separando dos grupos |
| `SMOTE` | Imbalanced-learn | Crear copias de fraudes para balancear el dataset |
| `import xgboost as xgb` | XGBoost | Algoritmo #3: el que gana la mayoría de competencias de ML |

> ✅ **Si no aparece ningún error en rojo** — todo está abierto y listo.


In [ ]:
# ── Herramientas generales ─────────────────────────────
import pandas as pd         # "pd" es el apodo corto que le ponemos (convención universal)
import numpy as np          # "np" es el apodo de numpy (también convención universal)
import warnings, os
warnings.filterwarnings('ignore')  # silenciar avisos no críticos — pantalla limpia

# ── Herramientas de Machine Learning (scikit-learn) ────
from sklearn.model_selection  import train_test_split    # dividir datos entrenamiento/prueba
from sklearn.preprocessing    import StandardScaler      # escalar números a la misma escala
from sklearn.metrics          import roc_auc_score, confusion_matrix  # medir el modelo
from sklearn.ensemble         import RandomForestClassifier  # Algoritmo #1
from sklearn.linear_model     import LogisticRegression      # Algoritmo #2

# ── Herramienta para datos desbalanceados ───────────────
from imblearn.over_sampling   import SMOTE  # crear fraudes sintéticos para balancear

# ── Algoritmo #3 ────────────────────────────────────────
import xgboost as xgb       # XGBoost — el campeón de las competencias de ML

print('✅ Todas las herramientas abiertas y listas.')
print('   Sin errores en rojo = perfecto, continúa al Paso 3 →')


✅ Todas las herramientas abiertas y listas.
   Sin errores en rojo = perfecto, continúa al Paso 3 →


## 📂 Paso 3 de 9 — Cargar los datos reales

---

### 🎯 Por qué hacemos esto

Un modelo de Machine Learning **no puede aprender de la nada**.  
Necesita ver muchos ejemplos reales con sus respuestas correctas antes de poder predecir.

**La analogía del detective:**  
Imagina que eres detective de fraude en un banco.  
Durante dos años guardaste en una libreta cada transacción que ocurrió.  
Al final de cada fila anotaste: *legítima* o *fraude*.

Eso es exactamente lo que vamos a cargar:  
**50,000 transacciones bancarias reales**, ya clasificadas por analistas del banco.  
El modelo va a estudiar esa libreta para aprender los patrones.

---

### 🔢 Los números importantes que verás

Cuando ejecutes esta celda, el output te mostrará algo como:

```
Total de transacciones:  50,000
De esas, son fraude:     492  (0.98%)
Es decir: de cada 102 compras, solo 1 es fraude
```

**¿Por qué es importante ese 0.98%?**

Ese es el **problema central** de detección de fraude.  
El fraude es rarísimo — menos del 1% de las transacciones.  
Encontrarlo entre tantas transacciones legítimas es como  
encontrar una aguja en un pajar.

Si el modelo dijera "legítima" para TODAS las transacciones,  
tendría un 99.02% de accuracy. Pero detectaría **0 fraudes**.  
Por eso la accuracy sola no es suficiente para medir este problema.

---

### ❓ ¿Por qué las columnas se llaman V1, V2... V28?

Las columnas originales del banco contenían datos privados:  
nombre del cliente, número de cuenta, comercio, etc.

Para proteger la privacidad de los clientes reales,  
el banco aplicó una transformación matemática (PCA)  
que convierte los datos originales en números abstractos.

**Nadie sabe qué significa V14 = -4.72 en términos reales.**  
Pero el modelo no necesita saberlo — solo necesita los números.  
Los patrones matemáticos son suficientes para detectar el fraude.

---

### 💬 Qué hace el código

Descarga la tabla desde GitHub, la carga en memoria,  
y muestra las primeras 3 filas para que puedas ver cómo luce.


In [ ]:
# URL del dataset en GitHub — 50,000 transacciones reales anonimizadas
DATASET_URL = 'https://raw.githubusercontent.com/juanmiguelsuero/fraudguard-ai/main/creditcard_mini.csv'

print('📥 Descargando datos desde GitHub...')
df = pd.read_csv(DATASET_URL)  # leer la tabla CSV y guardarla en 'df' (dataframe)

# ── Calcular estadísticas básicas ──────────────────────
total_real   = len(df)                           # total de filas en la tabla
fraudes_real = int(df['Class'].sum())            # Class=1 significa fraude, Class=0 legítima
ratio        = int((df['Class']==0).sum() // fraudes_real)  # cuántas legítimas por cada fraude

# ── Mostrar resumen ─────────────────────────────────────
print('=' * 55)
print('  LOS DATOS QUE VAMOS A USAR')
print('=' * 55)
print(f'  Total de transacciones:  {total_real:,}')
print(f'  De esas, son fraude:     {fraudes_real} ({fraudes_real/total_real*100:.2f}%)')
print(f'  Es decir: de cada {ratio} compras, solo 1 es fraude')
print('=' * 55)
print()
print('Así luce la tabla (algunas columnas seleccionadas):')

# Mostrar solo las columnas más interesantes
df[['Time', 'V1', 'V2', 'V14', 'V17', 'Amount', 'Class']].head(3)

# NOTA: 'Class' es la columna objetivo — lo que queremos predecir
# Class = 0 → transacción LEGÍTIMA
# Class = 1 → transacción FRAUDE


📥 Descargando datos desde GitHub...
  LOS DATOS QUE VAMOS A USAR
  Total de transacciones:  50,000
  De esas, son fraude:     492 (0.98%)
  Es decir: de cada 100 compras, solo 1 es fraude

Así luce la tabla (algunas columnas seleccionadas):


,Time,V1,V2,V14,V17,Amount,Class
0,71153.0,-1.987284,0.608930,0.468325,-0.018090,1.00,0
1,109575.0,-0.532974,0.756122,1.396922,-0.027192,11.27,0
2,59385.0,-7.626924,-6.976420,-5.281678,-4.442082,18.98,1


## ⚖️ Paso 4 de 9 — Escalar los números

---

### 🎯 Por qué hacemos esto

Hay un problema silencioso en los datos:  
**las variables tienen escalas completamente diferentes**.

- `Amount` puede ser $5,000
- `V14` puede ser -2.3
- `Time` puede ser 150,000 (segundos)

Si el modelo ve esos números sin preparación,  
**le va a dar más importancia al Amount solo porque es un número más grande**,  
aunque V14 sea mucho más relevante para detectar fraude.

Es como comparar el peso de una persona en kilos (75)  
con su temperatura en Celsius (37).  
Ambos son números, pero no tienen nada que ver entre sí.

**La solución:** StandardScaler convierte todos los valores  
para que tengan la misma escala — promedio cercano a 0 y rango similar.  
Así el modelo los puede comparar de forma justa.

---

### 💬 Qué hace el código

1. Crea el escalador
2. Lo aplica a `Amount` (el monto) y `Time` (la hora)
3. Separa la columna objetivo `Class` (lo que queremos predecir)
4. Separa las variables de entrada `X` de la variable a predecir `y`

**¿Por qué solo Amount y Time y no todas?**  
Las columnas V1 a V28 ya vienen pre-escaladas por el banco.  
Solo Amount y Time están en sus valores originales.


In [ ]:
# ── Escalar Amount y Time ──────────────────────────────
# StandardScaler: convierte los valores para que tengan promedio=0 y escala similar
scaler = StandardScaler()

# fit_transform = aprender la escala Y aplicarla al mismo tiempo
# [[columna]] — los dobles corchetes le dicen a pandas que queremos un DataFrame, no una Serie
df['Amount'] = scaler.fit_transform(df[['Amount']])
df['Time']   = scaler.fit_transform(df[['Time']])

# ── Separar X (entradas) e y (objetivo) ────────────────
# X = todo lo que el modelo VE — las 30 variables de entrada
# y = lo que el modelo tiene que PREDECIR — 0 (legítima) o 1 (fraude)
X = df.drop('Class', axis=1)  # todo MENOS la columna Class
y = df['Class']               # SOLO la columna Class

print('✅ Escalado completado.')
print()
print(f'  X tiene {X.shape[0]:,} filas y {X.shape[1]} columnas (variables de entrada)')
print(f'  y tiene {len(y):,} valores — 0=legítima, 1=fraude')
print()
print('  Primeros valores de y (la columna que queremos predecir):')
print( '  ' + str(y.value_counts().to_dict()))
print()
print('  Continúa al Paso 5 → (la regla de oro)')


✅ Escalado completado.

  X tiene 50,000 filas y 30 columnas (variables de entrada)
  y tiene 50,000 valores — 0=legítima, 1=fraude

  Primeros valores de y (la columna que queremos predecir):
  {0: 49508, 1: 492}

  Continúa al Paso 5 → (la regla de oro)


## ✂️ Paso 5 de 9 — Dividir los datos: Entrenamiento y Prueba

---

### 🎯 Por qué hacemos esto — LA REGLA DE ORO

Este es el paso más importante de todo el proceso.  
Y el que más gente hace mal.

**El problema:** si el modelo estudia TODAS las transacciones  
y después lo evaluamos con esas mismas transacciones —  
es como darle el examen al estudiante el día anterior  
y evaluarlo con exactamente ese mismo examen.

El modelo va a "acertar" todo porque ya vio las respuestas.  
Pero no aprendió nada real. Las métricas que obtengamos serían **mentira**.

**La solución:** dividir los datos en dos grupos antes de cualquier entrenamiento:
- **80% para entrenar** — el modelo estudia esto
- **20% para probar** — el modelo nunca vio esto; lo usamos solo para medir

El 20% de prueba es el "examen sorpresa". Si el modelo lo pasa — aprendió de verdad.

---

### ⚠️ La trampa de SMOTE

SMOTE va a crear fraudes sintéticos (en el Paso 5B).  
**SMOTE solo puede aplicarse al conjunto de entrenamiento — NUNCA al de prueba.**

¿Por qué? Porque el conjunto de prueba tiene que ser 100% datos reales.  
Si mezclamos fraudes sintéticos en la prueba, las métricas quedan contaminadas.

**La secuencia correcta siempre es:**
```
1. Separar datos en train/test  ← primero
2. Aplicar SMOTE solo al train  ← después
```
Si lo haces al revés, cualquier porcentaje de AUC que obtengas es un número inventado.

---

### 💬 El parámetro `stratify=y`

Sin `stratify`, podría pasar que todos los fraudes queden en el train  
y ninguno en el test — o viceversa.

`stratify=y` garantiza que **la proporción de fraudes sea igual en ambos grupos**.  
Si el dataset tiene 0.98% de fraudes, el train tendrá 0.98% y el test también.


In [ ]:
# ── División 80% entrenamiento / 20% prueba ────────────
X_train, X_test, y_train, y_test = train_test_split(
    X,              # variables de entrada
    y,              # variable objetivo
    test_size=0.2,  # 20% para prueba, 80% para entrenamiento
    stratify=y,     # mantener la misma proporción de fraudes en ambos grupos
    random_state=42 # semilla para reproducibilidad — siempre el mismo resultado
)

# ── Mostrar los tamaños ─────────────────────────────────
fraudes_train = int(y_train.sum())
fraudes_test  = int(y_test.sum())

print('✅ División completada.')
print()
print('  ┌─────────────────────────────────────────────────┐')
print(f'  │  ENTRENAMIENTO:  {len(X_train):,} filas  ({fraudes_train} fraudes)  │')
print(f'  │  PRUEBA:         {len(X_test):,}  filas  ({fraudes_test} fraudes)  │')
print('  └─────────────────────────────────────────────────┘')
print()
print('  IMPORTANTE:')
print('  El modelo va a estudiar SOLO el grupo de entrenamiento.')
print('  El grupo de prueba se guarda en un cajón cerrado.')
print('  Solo lo abrimos al final para medir — como un examen sorpresa.')
print()

# ── Aplicar SMOTE solo al train ─────────────────────────
# SMOTE = Synthetic Minority Over-sampling Technique
# Crea fraudes sintéticos matemáticamente a partir de los reales
# REGLA: solo aplicar al TRAIN — nunca al test
print('  Aplicando SMOTE al conjunto de entrenamiento...')
print(f'  Antes de SMOTE: {fraudes_train} fraudes en el train')

X_train, y_train = SMOTE(random_state=42).fit_resample(X_train, y_train)
n_por_clase = int((y_train == 0).sum())

print(f'  Después de SMOTE: {int(y_train.sum()):,} fraudes (sintéticos + reales)')
print(f'  Ahora hay {n_por_clase:,} ejemplos de cada clase → dataset balanceado')
print()
print('  El test permanece intacto con solo datos reales.')
print('  Continúa al Paso 6 → (el entrenamiento)')


✅ División completada.

  ┌─────────────────────────────────────────────────┐
  │  ENTRENAMIENTO:  40,000 filas  (394 fraudes)  │
  │  PRUEBA:         10,000  filas  (98 fraudes)  │
  └─────────────────────────────────────────────────┘

  IMPORTANTE:
  El modelo va a estudiar SOLO el grupo de entrenamiento.
  El grupo de prueba se guarda en un cajón cerrado.
  Solo lo abrimos al final para medir — como un examen sorpresa.

  Aplicando SMOTE al conjunto de entrenamiento...
  Antes de SMOTE: 394 fraudes en el train
  Después de SMOTE: 39,606 fraudes (sintéticos + reales)
  Ahora hay 39,606 ejemplos de cada clase → dataset balanceado

  El test permanece intacto con solo datos reales.
  Continúa al Paso 6 → (el entrenamiento)


## 🧠 Paso 6 de 9 — Entrenar los 3 modelos

---

### 🎯 Por qué hacemos esto

Llegó el momento que todo el proceso estaba preparando.  
**El modelo va a aprender.**

Vamos a entrenar 3 algoritmos distintos al mismo tiempo,  
porque cada uno tiene una forma diferente de encontrar patrones.  
Al final los comparamos y usamos los 3 juntos.

---

### 🌲 Algoritmo 1 — Random Forest (Bosque Aleatorio)

**Cómo funciona:** construye 20 árboles de decisión independientes.  
Cada árbol analiza los datos desde un ángulo diferente.  
Para tomar una decisión, los 20 árboles votan.  
La mayoría gana.

**La analogía:** un jurado de 20 expertos independientes.  
Cada uno analiza el caso por su cuenta.  
El veredicto es el de la mayoría.

**Sus fortalezas:** estable, difícil de engañar, funciona bien con pocos ajustes.

---

### ⚡ Algoritmo 2 — XGBoost (Extreme Gradient Boosting)

**Cómo funciona:** construye árboles en secuencia.  
Cada árbol nuevo corrige los errores del árbol anterior.  
El último árbol tiene acumulada toda la sabiduría de los anteriores.

**La analogía:** un equipo donde cada miembro revisa  
y corrige el trabajo del anterior.  
El resultado final es el más refinado.

**Sus fortalezas:** el más preciso de los tres, gana la mayoría de competencias de ML.  
`scale_pos_weight=20` le dice que los fraudes son 20 veces más raros que las legítimas — que los tome en cuenta.

---

### 📏 Algoritmo 3 — Regresión Logística

**Cómo funciona:** encuentra una línea matemática (en 30 dimensiones)  
que separa las transacciones fraudulentas de las legítimas.  
Cada variable tiene un peso. Si la suma de pesos supera el umbral — fraude.

**La analogía:** una balanza. Cada señal de riesgo pesa algo.  
Si el lado del fraude pesa más que el umbral → bloqueado.

**Sus fortalezas:** el más explicable. El banco puede justificar cada decisión ante un regulador.

---

### 💬 El parámetro `n_jobs=-1`

Le dice al algoritmo que use todos los núcleos del procesador.  
Si tu computadora tiene 4 núcleos, trabaja 4 veces más rápido.

### ⏱️ Tiempo estimado

Este paso puede tardar **30–60 segundos** según la velocidad de Colab.  
Mientras corre, el modelo está analizando las ~40,000 filas de entrenamiento.


In [ ]:
# ── Definir los 3 modelos ───────────────────────────────
modelos = {

    'Random Forest': RandomForestClassifier(
        n_estimators=20,   # 20 árboles de decisión que votan juntos
        random_state=42,   # reproducibilidad — siempre el mismo resultado
        n_jobs=-1          # usar todos los núcleos del procesador
    ),

    'XGBoost': xgb.XGBClassifier(
        n_estimators=50,         # 50 árboles en secuencia, cada uno corrige al anterior
        scale_pos_weight=20,     # los fraudes son ~20x más raros — que les dé más peso
        eval_metric='logloss',   # métrica interna de entrenamiento
        random_state=42,
        verbosity=0              # sin mensajes durante el entrenamiento
    ),

    'Logistica': LogisticRegression(
        class_weight='balanced', # ajustar automáticamente el peso de cada clase
        max_iter=1000,           # máximo 1000 iteraciones para encontrar la línea
        random_state=42
    ),

}

# ── Entrenar cada modelo ────────────────────────────────
print('⚙️  Entrenando los 3 modelos...')
print('   (esto puede tardar 30-60 segundos)')
print()

for i, (nombre, modelo) in enumerate(modelos.items(), 1):
    print(f'  [{i}/3] Entrenando {nombre}...')
    modelo.fit(X_train, y_train)  # FIT = el modelo estudia los datos de entrenamiento
    print(f'       ✓ {nombre} listo')

print()
print('✅ Los 3 modelos fueron entrenados.')
print()
print('  ¿Qué ocurrió durante el entrenamiento?')
print('  Cada modelo analizó las filas de entrenamiento,')
print('  buscó los patrones que separan fraudes de legítimas,')
print('  y guardó esos patrones internamente.')
print()
print('  Ahora vamos a ver si aprendieron bien → Paso 7')


⚙️  Entrenando los 3 modelos...
   (esto puede tardar 30-60 segundos)

  [1/3] Entrenando Random Forest...
       ✓ Random Forest listo
  [2/3] Entrenando XGBoost...
       ✓ XGBoost listo
  [3/3] Entrenando Logistica...
       ✓ Logistica listo

✅ Los 3 modelos fueron entrenados.

  ¿Qué ocurrió durante el entrenamiento?
  Cada modelo analizó las filas de entrenamiento,
  buscó los patrones que separan fraudes de legítimas,
  y guardó esos patrones internamente.

  Ahora vamos a ver si aprendieron bien → Paso 7


## 📊 Paso 7 de 9 — Evaluar los resultados

---

### 🎯 Por qué hacemos esto

El modelo ya aprendió. Ahora viene la **prueba de fuego**:  
¿qué tan bien funciona con transacciones que nunca vio?

Abrimos el cajón del 20% de prueba que guardamos en el Paso 5.  
Le damos al modelo esas transacciones — sin decirle cuáles son fraude.  
El modelo las clasifica solo. Después comparamos con la realidad.

---

### 📐 Las métricas que importan

**AUC-ROC** — la métrica principal para detección de fraude

- `1.00` = el modelo perfecto (detecta todo, sin errores)
- `0.98` = excelente (lo que esperamos obtener hoy)
- `0.50` = el modelo aleatorio (tirar una moneda al aire)

**Por qué no usamos Accuracy:**  
Si el modelo dijera "legítima" para todas las transacciones, tendría 99% de accuracy.  
Pero detectaría **0 fraudes**. El AUC-ROC mide si el modelo realmente distingue.

---

### 🗃️ La Matriz de Confusión — leer los 4 números

Cuando el modelo evalúa el conjunto de prueba, hay 4 resultados posibles:

```
                 PREDICCIÓN DEL MODELO
                 Legítima  |  Fraude
               ┌───────────┬──────────┐
REALIDAD  Leg. │    TP     │   FP     │  ← legítimas reales
               ├───────────┼──────────┤
          Fraud│    FN     │   TP     │  ← fraudes reales
               └───────────┴──────────┘
```

- **TP (True Positives):** fraudes reales que el modelo detectó ✓
- **FP (False Positives):** legítimas que el modelo bloqueó por error ✗
- **FN (False Negatives):** fraudes reales que el modelo dejó pasar ✗ ← el más costoso
- **TN (True Negatives):** legítimas que el modelo aprobó correctamente ✓

**Recall** = TP / (TP + FN) = de todos los fraudes reales, ¿cuántos detectó?  
Un recall de 85% significa que detectó 85 de cada 100 fraudes reales.

---

### 🎚️ El umbral — puedes cambiarlo

`UMBRAL = 0.3` significa: si el modelo dice ≥30% de probabilidad de fraude → bloquear.

Prueba cambiar este número:
- `UMBRAL = 0.1` → más agresivo, detecta más fraudes pero bloquea más clientes legítimos
- `UMBRAL = 0.5` → más conservador, menos bloqueos erróneos pero algunos fraudes pasan


In [ ]:
# ── Umbral de decisión ─────────────────────────────────
# Si el modelo dice que hay >= UMBRAL de probabilidad de fraude → BLOQUEADA
# Puedes cambiar este número y ejecutar de nuevo para ver el efecto
UMBRAL = 0.3

# ── Evaluar cada modelo ─────────────────────────────────
total_test    = int((y_test == 1).sum())   # fraudes reales en el test
total_leg     = int((y_test == 0).sum())   # legítimas reales en el test

print(f'  Conjunto de prueba: {total_leg:,} legítimas + {total_test} fraudes reales')
print(f'  Umbral de decisión: {UMBRAL} ({UMBRAL*100:.0f}% de probabilidad = bloqueada)')
print('=' * 60)
print()

for nombre, modelo in modelos.items():

    # predict_proba devuelve [prob_legítima, prob_fraude] para cada transacción
    # tomamos [:, 1] para quedarnos solo con la probabilidad de fraude
    proba = modelo.predict_proba(X_test)[:, 1]

    # Aplicar el umbral: si prob_fraude >= UMBRAL → predecir como fraude (1)
    pred = (proba >= UMBRAL).astype(int)

    # Calcular métricas
    auc = roc_auc_score(y_test, proba)       # AUC-ROC (usa probabilidades)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    print(f'  [{nombre}]')
    print(f'  {"─" * 54}')
    print(f'  DETECTO  {int(tp):>4} fraudes de {total_test}  ✓  (recall: {tp/(tp+fn)*100:.1f}%)')
    print(f'  PERDIO   {int(fn):>4} fraudes            ✗  (dinero que se escapó)')
    print(f'  BLOQUEO  {int(fp):>4} legítimas por error ✗  (clientes molestos)')
    print(f'  AUC-ROC: {auc:.4f}   (1.0 = perfecto, 0.5 = azar puro)')
    print()

print('=' * 60)
print()
print('  💡 Prueba cambiar UMBRAL = 0.1 o UMBRAL = 0.5 y ejecuta de nuevo.')
print('     Observa cómo cambia el número de fraudes detectados vs. clientes bloqueados.')


  Conjunto de prueba: 9,902 legítimas + 98 fraudes reales
  Umbral de decisión: 0.3 (30% de probabilidad = bloqueada)

  [Random Forest]
  ──────────────────────────────────────────────────────
  DETECTO    85 fraudes de 98  ✓  (recall: 86.7%)
  PERDIO     13 fraudes            ✗  (dinero que se escapó)
  BLOQUEO    34 legítimas por error ✗  (clientes molestos)
  AUC-ROC: 0.9676   (1.0 = perfecto, 0.5 = azar puro)

  [XGBoost]
  ──────────────────────────────────────────────────────
  DETECTO    87 fraudes de 98  ✓  (recall: 88.8%)
  PERDIO     11 fraudes            ✗  (dinero que se escapó)
  BLOQUEO    89 legítimas por error ✗  (clientes molestos)
  AUC-ROC: 0.9725   (1.0 = perfecto, 0.5 = azar puro)

  [Logistica]
  ──────────────────────────────────────────────────────
  DETECTO    92 fraudes de 98  ✓  (recall: 93.9%)
  PERDIO      6 fraudes            ✗  (dinero que se escapó)
  BLOQUEO   521 legítimas por error ✗  (clientes molestos)
  AUC-ROC: 0.9750   (1.0 = perfecto, 0.5 = aza

## 🔍 Paso 8 de 9 — Ver transacciones reales siendo juzgadas

---

### 🎯 Por qué hacemos esto

Los números del Paso 7 son buenos para medir.  
Pero no muestran cómo funciona el modelo transacción por transacción.

Ahora vamos a tomar **6 transacciones reales** del conjunto de prueba  
(que el modelo nunca vio durante el entrenamiento)  
y ver exactamente qué decidió el modelo para cada una.

---

### 🎬 Lo que verás

```
Realidad: FRAUDE real
Modelo:   FRAUDE  ⚠  100.0% [████████████████████] ALTO   ✓ CORRECTO

Realidad: LEGÍTIMA real  
Modelo:   LEGÍTIMA ✓    2.0% [░░░░░░░░░░░░░░░░░░░░] BAJO   ✓ CORRECTO
```

La barra de progreso muestra visualmente la probabilidad que el modelo calculó.  
`████` llena = muy seguro que es fraude.  
`░░░░` vacía = muy seguro que es legítima.

**¿Y si aparece un error?**  
Que el modelo se equivoque en alguna transacción es normal y esperado.  
Un modelo de ML no tiene que ser perfecto — tiene que ser mejor que las reglas manuales.

---

### 💡 Prueba cambiar `.head(3)` por `.sample(3)`

Con `.head(3)` siempre ves las mismas 3 transacciones.  
Con `.sample(3)` ves 3 transacciones **aleatorias** distintas cada vez que ejecutas.  
Ejecútalo varias veces y observa cómo responde el modelo a distintos casos.


In [ ]:
# ── Seleccionar transacciones del conjunto de prueba ───
# .head(3) = las 3 primeras filas del test
# Prueba cambiar a .sample(3) para ver 3 aleatorias distintas cada vez

fraudes_test_df = X_test[y_test == 1].head(3)    # 3 fraudes reales del test
legitimas_test_df = X_test[y_test == 0].head(3)  # 3 legítimas reales del test

# Combinarlas y mezclarlas
muestra = pd.concat([fraudes_test_df, legitimas_test_df])
etiquetas = (['FRAUDE real'] * 3) + (['LEGÍTIMA real'] * 3)

print('=' * 60)
print("  TRANSACCIONES QUE EL MODELO NUNCA VIO")
print("  'Realidad' = lo que realmente era")
print("  'Modelo'   = lo que el modelo predijo")
print('=' * 60)

for (idx, fila), etiqueta_real in zip(muestra.iterrows(), etiquetas):
    fila_df = fila.to_frame().T  # convertir la fila al formato que espera el modelo

    # Usar Random Forest para este ejemplo
    score = modelos['Random Forest'].predict_proba(fila_df)[0][1]
    pct   = score * 100

    # Construir visualización de la barra
    barra  = '█' * int(score * 20) + '░' * (20 - int(score * 20))
    nivel  = 'ALTO  ' if pct >= 70 else 'MEDIO ' if pct >= 30 else 'BAJO  '
    pred   = 'FRAUDE  ⚠' if score >= UMBRAL else 'LEGÍTIMA ✓'

    # ¿El modelo acertó?
    acierto = '✓ CORRECTO' if (
        (score >= UMBRAL and 'FRAUDE'   in etiqueta_real) or
        (score <  UMBRAL and 'LEGÍTIMA' in etiqueta_real)
    ) else '✗ ERROR'

    print(f'\n  Realidad: {etiqueta_real}')
    print(f'  Modelo:   {pred}  {pct:5.1f}% [{barra}] {nivel} {acierto}')

print()
print('  💡 Cambia .head(3) por .sample(3) para ver transacciones distintas cada vez.')


  TRANSACCIONES QUE EL MODELO NUNCA VIO
  'Realidad' = lo que realmente era
  'Modelo'   = lo que el modelo predijo

  Realidad: FRAUDE real
  Modelo:   FRAUDE  ⚠  100.0% [████████████████████] ALTO   ✓ CORRECTO

  Realidad: FRAUDE real
  Modelo:   FRAUDE  ⚠  100.0% [████████████████████] ALTO   ✓ CORRECTO

  Realidad: FRAUDE real
  Modelo:   FRAUDE  ⚠  100.0% [████████████████████] ALTO   ✓ CORRECTO

  Realidad: LEGÍTIMA real
  Modelo:   LEGÍTIMA ✓    0.0% [░░░░░░░░░░░░░░░░░░░░] BAJO   ✓ CORRECTO

  Realidad: LEGÍTIMA real
  Modelo:   LEGÍTIMA ✓    0.0% [░░░░░░░░░░░░░░░░░░░░] BAJO   ✓ CORRECTO

  Realidad: LEGÍTIMA real
  Modelo:   LEGÍTIMA ✓    0.0% [░░░░░░░░░░░░░░░░░░░░] BAJO   ✓ CORRECTO

  💡 Cambia .head(3) por .sample(3) para ver transacciones distintas cada vez.


## 🧪 Paso 9 de 9 — Crea tu propia transacción

---

### 🎯 Por qué hacemos esto

Ahora **tú eres el inventor de la transacción**.  
El modelo decide si es fraude o legítima.  
Nadie le dice la respuesta — él la deduce solo  
de los patrones que aprendió en el Paso 6.

---

### 🎮 Los valores que puedes cambiar

| Variable | Qué representa | Ejemplo normal | Ejemplo fraude |
|----------|---------------|---------------|----------------|
| `Time`   | Hora (segundos desde medianoche) | `43200` = 12pm | `10800` = 3am |
| `Amount` | Monto en dólares | `9.99` | `1.00` |
| `V14`    | Patrón matemático clave #14 | `0.31` | `-9.53` |
| `V17`    | Patrón matemático clave #17 | `-0.30` | `-7.74` |

---

### 🔬 El experimento que más sorprende a todo el mundo

1. La **Transacción 1** tiene `V14=0.31` y `V17=-0.30` → el modelo dice **LEGÍTIMA**
2. Cambia **solo** `V14` a `-9.53` y `V17` a `-7.74`
3. Ejecuta la celda de nuevo con `Shift + Enter`
4. La misma transacción, con el mismo monto y la misma hora → ahora dice **FRAUDE**

**¿Por qué?**  
V14=-9.53 y V17=-7.74 son los valores que aparecen en la gran mayoría  
de fraudes reales del dataset. El modelo reconoció ese patrón matemático.

Cambias un solo número → el modelo cambia de opinión completamente.  
Eso es exactamente lo que hace el banco en **150 milisegundos**  
cada vez que pasas tu tarjeta.

---

### 💡 Más experimentos para probar

- ¿Qué pasa si pones `Amount = 0.01` (un centavo) a las 3am?
- ¿Y si pones `Time = 10800` (3am) con V14 y V17 normales?
- ¿Cuál de las dos variables tiene más peso — V14 o V17?


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║   EDITA LOS VALORES AQUÍ  →  EJECUTA DE NUEVO       ║
# ║   Shift + Enter para ejecutar                        ║
# ╚══════════════════════════════════════════════════════╝

# ── TRANSACCIÓN 1 — Empieza como LEGÍTIMA ──────────────
# EXPERIMENTO: cambia V14 a -9.53 y V17 a -7.74 → se convierte en FRAUDE
transaccion_1 = pd.DataFrame([{
    'Time':  32400,    # 9:00am (hora × 3600 para convertir a segundos)
    'V1':    1.19,  'V2':   0.26,  'V3':   0.17,  'V4':   0.45,
    'V5':    0.06,  'V6':  -0.08,  'V7':  -0.07,  'V8':   0.09,
    'V9':    0.31,  'V10':  0.17,  'V11':  0.14,  'V12':  0.12,
    'V13':   0.05,
    'V14':   0.31,     # ← CÁMBIAME a -9.53 para ver el cambio
    'V15':   0.02,  'V16': -0.05,
    'V17':  -0.30,     # ← CÁMBIAME a -7.74 para completar el cambio
    'V18':   0.01,  'V19':  0.10,  'V20':  0.02,
    'V21':   0.01,  'V22':  0.02,  'V23':  0.00,  'V24':  0.00,
    'V25':   0.01,  'V26':  0.01,  'V27':  0.00,  'V28':  0.00,
    'Amount': 9.99,    # ← monto en dólares — prueba cambiarlo
}])[X.columns]

# ── TRANSACCIÓN 2 — Ejemplo de FRAUDE real ─────────────
# Copiada de un fraude real del dataset
# V14 y V17 muy negativos = señal clásica de fraude
transaccion_2 = pd.DataFrame([{
    'Time':   406,     # 12:06am — casi medianoche
    'V1':   -3.04,  'V2':   1.35,  'V3':  -3.47,  'V4':   0.74,
    'V5':   -0.40,  'V6':   0.83,  'V7':  -1.37,  'V8':   0.14,
    'V9':   -1.46,  'V10': -2.06,  'V11':  2.10,  'V12': -2.63,
    'V13':   0.48,
    'V14':  -9.53,     # ← muy negativo = señal FUERTE de fraude
    'V15':   0.18,  'V16': -0.26,
    'V17':  -7.74,     # ← muy negativo = señal FUERTE de fraude
    'V18':   0.49,  'V19':  0.01,  'V20':  0.24,
    'V21':   0.75,  'V22': -0.56,  'V23': -0.17,  'V24':  0.06,
    'V25':  -0.06,  'V26': -0.01,  'V27':  0.20,  'V28':  0.14,
    'Amount': 1.00,    # ← $1 — compra de prueba para verificar que la tarjeta funciona
}])[X.columns]

# ── Evaluar con los 3 modelos ───────────────────────────
mis_transacciones = [
    ('TRANSACCIÓN 1 — $9.99 a las 09:00h', transaccion_1),
    ('TRANSACCIÓN 2 — $1.00 a las 00:06h', transaccion_2),
]

print(f'{"=" * 60}')
print(f'  TUS TRANSACCIONES — los 3 modelos opinan')
print(f'  Edita los valores de arriba y ejecuta de nuevo')
print(f'{"=" * 60}')

for desc, fila_df in mis_transacciones:
    print(f'\n  {desc}')
    print(f'  {"─" * 56}')
    for nombre, modelo in modelos.items():
        score  = modelo.predict_proba(fila_df)[0][1]
        pct    = score * 100
        pred   = 'FRAUDE  ⚠' if score >= UMBRAL else 'LEGÍTIMA ✓'
        barra  = '█' * int(score * 20) + '░' * (20 - int(score * 20))
        nivel  = 'ALTO  ' if pct >= 70 else 'MEDIO ' if pct >= 30 else 'BAJO  '
        print(f'    {nombre:15} {pred:12} {pct:5.1f}% [{barra}] {nivel}')

print(f'\n{"─" * 60}')
print(f'  0% – 29%   → BAJO   → el banco APRUEBA la compra automáticamente')
print(f'  30% – 69%  → MEDIO  → el banco MANDA un SMS al cliente para confirmar')
print(f'  70% – 100% → ALTO   → el banco BLOQUEA al instante y llama después')
print(f'{"=" * 60}')


---

## 🎉 ¡Lo construiste!

Acabas de crear el mismo tipo de sistema que usa un banco real.  
**En 9 pasos. Con datos reales. En tu computadora.**

---

### 🔁 Más experimentos para explorar

**Cambiar el umbral (Paso 7):**
```python
UMBRAL = 0.1   # más agresivo — detecta más fraudes, más clientes bloqueados
UMBRAL = 0.5   # más conservador — menos bloqueos, algunos fraudes pasan
```

**Ver transacciones aleatorias (Paso 8):**  
Cambia `.head(3)` por `.sample(3)` y ejecuta varias veces.

**El experimento del V14 (Paso 9):**  
Cambia `V14` de `0.31` a `-9.53` en la Transacción 1.  
La misma transacción pasa de LEGÍTIMA a FRAUDE.  
Un solo número. Eso es lo que el banco calcula en 150 milisegundos.

---

### 📚 Lo que aprendiste hoy

| Concepto | Lo que significa en la vida real |
|----------|----------------------------------|
| Machine Learning | Enseñarle a una computadora con ejemplos, no con reglas |
| Dataset desbalanceado | El fraude es rarísimo — encontrarlo es la aguja en el pajar |
| SMOTE | Crear ejemplos sintéticos para que el modelo aprenda mejor |
| Train/Test split | La regla de oro — el modelo nunca puede ver el examen antes |
| AUC-ROC | La métrica honesta — no se puede engañar con "legítima para todo" |
| Umbral | No es el algoritmo quien decide — es el director de riesgos del banco |
| V14, V17 | Los patrones matemáticos que distinguen el fraude — invisibles para humanos |

---

*🛡️ FraudGuard AI · HackConRD*  
*github.com/juanmiguelsuero/fraudguard-ai*
